# Pipeline
## General settings

In [2]:
import random
import sys
import numpy as np
import pandas as pd
import torch

from functools import partial
from pathlib import Path
from pycocotools.coco import COCO
from torch.utils.data import DataLoader
from transformers import RTDetrImageProcessor

PROJECT_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
COMPETITION = "multiple-object-video-detection-hw-2-ucu-ai-2026"
DIRECTORY = INPUT_ROOT / COMPETITION

if not (DIRECTORY / "train" / "annotations_coco.json").is_file():
    data_roots = sorted({
        path.parent.parent
        for path in INPUT_ROOT.rglob("annotations_coco.json")
        if path.parent.name == "train"
    })

    DIRECTORY = data_roots[0]

INFO_TRAIN = DIRECTORY / "train" / "annotations_coco.json"
INFO_VAL = DIRECTORY / "val" / "annotations_coco.json"
INFO_TEST = DIRECTORY / "test_images.csv"

required_scripts = {
    "data_prep.py",
    "dataset_helpers.py",
    "model_builders.py",
    "torch_model.py",
    "yolo26_model.py",
    "rdtdetr_model.py",
    "experiment_runner.py",
    "experiment_results.py",
}

script_dirs = sorted({
    path.parent for path in INPUT_ROOT.rglob("rdtdetr_model.py")
    if path.parent.name == "scripts"
    and all(
        (path.parent / filename).is_file()
        for filename in required_scripts
    )
})


scripts_parent = str(script_dirs[0].parent)
if scripts_parent not in sys.path:
    sys.path.insert(0, scripts_parent)

from scripts.data_prep import load_annotations
from scripts.dataset_helpers import DetectionDataset, RTDetrCollator
from scripts.model_builders import build_rtdetr
from scripts.experiment_runner import run_rtdetr
from scripts.experiment_results import model_training_results_final

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
EPOCHS = 12
BATCH_SIZE = 2
FRAME_STEP = 4
CHECKPOINT = "PekingU/rtdetr_r50vd"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Data preparation


In [4]:
processor = RTDetrImageProcessor.from_pretrained(CHECKPOINT)

coco_train = COCO(str(INFO_TRAIN))
coco_valid = COCO(str(INFO_VAL))
train_frames, train_boxes = load_annotations(coco_train, split="train")
valid_frames, valid_boxes = load_annotations(coco_valid, split="val")

train_annotations_by_image = {image_id: group for image_id, group in train_boxes.groupby("image_id")}
valid_annotations_by_image = {image_id: group for image_id, group in valid_boxes.groupby("image_id")}

sampled_train_frames = (train_frames.sort_values(["video", "frame_id"])
                        .groupby("video", group_keys=False).nth[::FRAME_STEP].reset_index(drop=True))

category_ids = sorted(coco_valid.getCatIds())
cat_to_label = {category_id: index for index, category_id in enumerate(category_ids)}
id_to_label = {index: coco_valid.cats[category_id]["name"] for category_id, index in cat_to_label.items()}
label_to_id = {name: index for index, name in id_to_label.items()}

preprocessor_config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

loading annotations into memory...
Done (t=6.09s)
creating index...
index created!
loading annotations into memory...
Done (t=0.63s)
creating index...
index created!


In [5]:
train_dataset = DetectionDataset(frames=sampled_train_frames, directory=DIRECTORY,
    annotations_by_image=train_annotations_by_image,
    empty_annotations=train_boxes.iloc[:0].copy(), validation=False, 
    horizontal=True, vertical=False, contrast=True, saturation=True
)

valid_dataset = DetectionDataset(
    frames=valid_frames, directory=DIRECTORY,
    annotations_by_image=valid_annotations_by_image,
    empty_annotations=valid_boxes.iloc[:0].copy(), validation=True
)

collator = RTDetrCollator(processor, cat_to_label)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collator)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collator)

In [6]:
model_kwargs = {"CHECKPOINT": CHECKPOINT, "id_to_label": id_to_label,
                "label_to_id": label_to_id, "num_queries": 200}

train_kwargs = {"processor": processor, "train_loader": train_loader,
                "valid_loader": valid_loader, "annotations_path": INFO_VAL,
                "device": device, "seed": SEED, "epochs": EPOCHS,
                "lr": 1e-4, "weight_decay": 1e-4, "max_hours": 7}

## Another data augmentation approach, increased number of queries and training budget

In [7]:
queries_histories, queries_results = run_rtdetr(**train_kwargs, model_method=partial(build_rtdetr, **model_kwargs,
                            matcher_bbox_cost=7.0), output_dir=PROJECT_ROOT / "outputs" / "rtdetr_queries",
                            lr_scheduler=True, step_size=4)

loading annotations into memory...
Done (t=0.45s)
creating index...
index created!


config.json:   0%|          | 0.00/5.11k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  172MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/764 [00:00<?, ?it/s]

[transformers] RTDetrForObjectDetection LOAD REPORT from: PekingU/rtdetr_r50vd
Key                                                 | Status   |                                                                                        
----------------------------------------------------+----------+----------------------------------------------------------------------------------------
model.enc_score_head.bias                           | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80]) vs model:torch.Size([8])          
model.decoder.class_embed.{0, 1, 2, 3, 4, 5}.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80, 256]) vs model:torch.Size([8, 256])
model.enc_score_head.weight                         | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80, 256]) vs model:torch.Size([8, 256])
model.decoder.class_embed.{0, 1, 2, 3, 4, 5}.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80]) vs model:torch.Size([8])          
mod

Loading and preparing results...
DONE (t=2.39s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=121.27s).
Accumulating evaluation results...
DONE (t=5.32s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.175
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.349
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.152
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.073
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.161
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.338
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.083
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.241
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.316
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=

### Results

In [8]:
queries_results_wrap = model_training_results_final(PROJECT_ROOT / "outputs" / "rtdetr_queries")

Training history


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,17.815433,0.000100,0.175416,0.349005,0.073167
1,2,15.082941,0.000100,0.204482,0.392652,0.074937
2,3,14.383378,0.000100,0.193552,0.376385,0.074431
3,4,13.884990,0.000100,0.195762,0.376848,0.083007
4,5,12.981619,0.000010,0.241460,0.445040,0.073886
5,6,12.719409,0.000010,0.257927,0.463701,0.081673
6,7,12.527769,0.000010,0.248560,0.448173,0.076538
7,8,12.395668,0.000010,0.244922,0.441552,0.069847
8,9,12.262281,0.000001,0.249393,0.447716,0.075716
9,10,12.230590,0.000001,0.249880,0.447819,0.075260



Best validation results — epoch 6


,mAP,AP50,AP_small
5,0.257927,0.463701,0.081673
